<a href="https://colab.research.google.com/github/v-yunusic/REPO-PYTHON-VY/blob/main/two-centuries-um/Notebook/ultramarathon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PENDIENTE

* Corregir calculo de tiempo rendimiento atleta, velocidad prom. está en km o millas?
* Estandarizar rendimiento de atleta a tiempo.

# PIP

In [ ]:
pip install gdown

# LIBRERIAS

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as st
import sympy as sp
import pandas as pd
import seaborn as sns
import re # RegEx
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import gdown
import gc

#========= KAGGLE =========#
import kagglehub

#========= OS =========#
import os

print('Librerias importadas!\n'+'-'*60)

Librerias importadas!
------------------------------------------------------------


# PREPROCESAMIENTO

In [ ]:
countries = None

In [ ]:
# https://drive.google.com/file/d/1z1JdNc9hZXbNxHaATZMrD_QcMR5qqMQG/view?usp=sharing

try:
  if not countries:
    file_id = '1z1JdNc9hZXbNxHaATZMrD_QcMR5qqMQG'
    url = f'https://drive.google.com/uc?id={file_id}'

    gdown.download(url, "UM_COUNTRIES.csv", quiet=True, use_cookies=False)

    countries = pd.read_csv('UM_COUNTRIES.csv', sep=',')
    print('\nDATAFRAME CARGADO!')
except:
  print(f'NO SE HA CARGADO DATAFRAME')


DATAFRAME CARGADO!


In [ ]:
countries['Country'].unique()

array(['CHI', 'ARG', 'COL', 'RUS', 'SLO', 'HUN', 'GER', 'USA', 'CAN',
       'MEX', 'ITA', 'JPN', 'BLR', 'NED', 'BEL', 'FRA', 'GBR', 'CZE',
       'SRB', 'TPE', 'NZL', 'AUS', 'SGP', 'HKG', 'CHN', 'POL', 'LUX',
       'SUI', 'ROU', 'CRC', 'NGR', 'ESP', 'PAR', 'BRA', 'ISR', 'AUT',
       'SWE', 'UKR', 'SVK', 'MDA', 'UAE', 'DEN', 'THA', 'MAS', 'BUL',
       'GRE', 'ALB', 'SMR', 'PER', 'POR', 'CMR', 'IRL', 'LAT', 'ECU',
       'GUA', 'BRU', 'IND', 'INA', 'ALG', 'LIE', 'CRO', 'GUY', 'IRI',
       'PHI', 'URU', 'NOR', 'MAR', 'NCA', 'VIE', 'VEN', 'RSA', 'GAB',
       'TUR', 'LTU', 'FIN', 'TUN', 'QAT', 'CYP', 'ZIM', 'MOZ', 'LES',
       'MAW', 'BOT', 'NAM', 'SWZ', 'ZAM', 'MRI', 'KOR', 'CAM', 'SRI',
       'HON', 'BOL', 'PUR', 'GRL', 'UZB', 'BIH', 'MNE', 'MKD', 'MAC',
       'MDV', 'MGL', 'EST', 'JOR', 'UGA', 'NEP', 'EGY', 'KEN', 'JAM',
       'ESA', 'MLT', 'CAY', 'AND', 'DOM', 'HAI', 'ANG', 'KOS', 'ISL',
       'ETH', 'OMA', 'KAZ', 'AZE', 'FIJ', 'CHA', 'LBA', 'CPV', 'MAD',
       'MYA', 'AFG',

In [ ]:
# PETICION DE DATA HACIA KAGGLE

path = kagglehub.dataset_download("aiaiaidavid/the-big-dataset-of-ultra-marathon-running")
print('DATASET DE KAGGLE CARGADO!')
print(path)

Using Colab cache for faster access to the 'the-big-dataset-of-ultra-marathon-running' dataset.
DATASET DE KAGGLE CARGADO!
/kaggle/input/the-big-dataset-of-ultra-marathon-running


In [ ]:
size = 0
width = 0
data_path = path+'/TWO_CENTURIES_OF_UM_RACES.csv'

try:
  for c in pd.read_csv(data_path,
                      chunksize=10000):
    if not c.empty:
      size += c.shape[0]
      width = c.shape[1]

  print(f'Cantidad de filas:\n{size}')
  print(f'Cantidad de columnas:\n{width}')
except Exception as e:
  print(e)

Cantidad de filas:
7461195
Cantidad de columnas:
13


##### ITERABLE

In [ ]:
#pd.read_csv('/root/.cache/kagglehub/datasets/aiaiaidavid/the-big-dataset-of-ultra-marathon-running/versions/2/TWO_CENTURIES_OF_UM_RACES.csv',
                     #chunksize=10000):

In [ ]:
pais = 'usa'.upper()

##### SEPARAR POR PAIS DE EVENTO

In [ ]:
df = None

In [ ]:
try:
  if not df or df.empty:
    for c in pd.read_csv(data_path,
                     chunksize=10000):
      filtrado = c.loc[c['Event name'].str.find(pais) > -1]
      if not c.empty:
        df = pd.concat([df, filtrado])

    print('\nDATAFRAME GENERADO!')
except:
  print('\nNO SE HA GENERADO DATAFRAME')


DATAFRAME GENERADO!


In [ ]:
df.head(2)

,Year of event,Event dates,Event name,Event distance/length,Event number of finishers,Athlete performance,Athlete club,Athlete country,Athlete year of birth,Athlete gender,Athlete age category,Athlete average speed,Athlete ID
55,2018,06.01.2018,Yankee Springs 50 Mile Winter Challenge (USA),50mi,9,9:53:05 h,"*Middleville, MI",USA,1983.0,M,M23,8.141,55
56,2018,06.01.2018,Yankee Springs 50 Mile Winter Challenge (USA),50mi,9,11:09:35 h,"*Waterloo, ON",CAN,1977.0,F,W40,7.211,56


##### SEPARAR POR PAÍS DE ATLETA

In [ ]:
try:
  if not df:
    for c in pd.read_csv('/root/.cache/kagglehub/datasets/aiaiaidavid/the-big-dataset-of-ultra-marathon-running/versions/2/TWO_CENTURIES_OF_UM_RACES.csv',
                     chunksize=10000):
      filtrado = c[c['Athlete country'] == pais]
      if not c.empty:
        df = pd.concat([df, filtrado])
    print('\nDATAFRAME GENERADO!')
except:
  print('\nNO SE HA GENERADO DATAFRAME')


NO SE HA GENERADO DATAFRAME


In [ ]:
df.head(2)

,Year of event,Event dates,Event name,Event distance/length,Event number of finishers,Athlete performance,Athlete club,Athlete country,Athlete year of birth,Athlete gender,Athlete age category,Athlete average speed,Athlete ID
55,2018,06.01.2018,Yankee Springs 50 Mile Winter Challenge (USA),50mi,9,9:53:05 h,"*Middleville, MI",USA,1983.0,M,M23,8.141,55
56,2018,06.01.2018,Yankee Springs 50 Mile Winter Challenge (USA),50mi,9,11:09:35 h,"*Waterloo, ON",CAN,1977.0,F,W40,7.211,56


In [ ]:
df.columns.tolist()

['Year of event',
 'Event dates',
 'Event name',
 'Event distance/length',
 'Event number of finishers',
 'Athlete performance',
 'Athlete club',
 'Athlete country',
 'Athlete year of birth',
 'Athlete gender',
 'Athlete age category',
 'Athlete average speed',
 'Athlete ID']

In [ ]:
print(f'Cantidad de filas:\n{df.shape[0]}')
print(f'Cantidad de columnas:\n{df.shape[1]}')

Cantidad de filas:
1408416
Cantidad de columnas:
13


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1408416 entries, 55 to 7461185
Data columns (total 13 columns):
 #   Column                     Non-Null Count    Dtype  
---  ------                     --------------    -----  
 0   Year of event              1408416 non-null  int64  
 1   Event dates                1408416 non-null  object 
 2   Event name                 1408416 non-null  object 
 3   Event distance/length      1408416 non-null  object 
 4   Event number of finishers  1408416 non-null  int64  
 5   Athlete performance        1408416 non-null  object 
 6   Athlete club               1096771 non-null  object 
 7   Athlete country            1408416 non-null  object 
 8   Athlete year of birth      1387458 non-null  float64
 9   Athlete gender             1408416 non-null  object 
 10  Athlete age category       1387838 non-null  object 
 11  Athlete average speed      1408236 non-null  object 
 12  Athlete ID                 1408416 non-null  int64  
dtypes: float64(1), i

In [ ]:
df['Event dates'].unique()

array(['06.01.2018', '01.01.2018', '24.03.2018', ..., '04.02.1995',
       '21.01.1995', '07.01.1995'], dtype=object)

In [ ]:
df['Event dates'] = df['Event dates'].apply(lambda x: x[-10:] if len(str(x)) > 10 else x)
df['Event dates'].head()

,Event dates
55,06.01.2018
56,06.01.2018
57,06.01.2018
58,06.01.2018
59,06.01.2018


In [ ]:
df['Event dates'] = pd.to_datetime(df['Event dates'], errors='coerce')
df['Event dates'].loc[df['Event dates'].isna() == True].shape[0]

0

In [ ]:
if df['Event dates'].loc[df['Event dates'].isna() == True].shape[0]/df.shape[0] < 0.05:
  df = df.loc[df['Event dates'].isna() == False]

In [ ]:
df.head(3)

,Year of event,Event dates,Event name,Event distance/length,Event number of finishers,Athlete performance,Athlete club,Athlete country,Athlete year of birth,Athlete gender,Athlete age category,Athlete average speed,Athlete ID
55,2018,2018-01-06,Yankee Springs 50 Mile Winter Challenge (USA),50mi,9,9:53:05 h,"*Middleville, MI",USA,1983.0,M,M23,8.141,55
56,2018,2018-01-06,Yankee Springs 50 Mile Winter Challenge (USA),50mi,9,11:09:35 h,"*Waterloo, ON",CAN,1977.0,F,W40,7.211,56
57,2018,2018-01-06,Yankee Springs 50 Mile Winter Challenge (USA),50mi,9,11:33:00 h,"*Kitchener, ON",CAN,1976.0,M,M40,6.967,57


In [ ]:
df['Event dates'].unique()

<DatetimeArray>
['2018-01-06 00:00:00', '2018-01-01 00:00:00', '2018-03-24 00:00:00',
 '2018-03-25 00:00:00', '2018-03-26 00:00:00', '2018-03-23 00:00:00',
 '2018-03-18 00:00:00', '2018-03-17 00:00:00', '2018-03-16 00:00:00',
 '2018-03-11 00:00:00',
 ...
 '1995-03-18 00:00:00', '1995-03-11 00:00:00', '1995-03-04 00:00:00',
 '1995-02-25 00:00:00', '1995-02-18 00:00:00', '1995-02-11 00:00:00',
 '1995-02-05 00:00:00', '1995-02-04 00:00:00', '1995-01-21 00:00:00',
 '1995-01-07 00:00:00']
Length: 4660, dtype: datetime64[ns]

In [ ]:
df['Athlete age'] = df['Event dates'].dt.year - df['Athlete year of birth']
df['Athlete age'].isna().sum()
df.loc[(df['Athlete year of birth'].isna()) & (df['Athlete age category'].isna())].shape[0]/df.shape[0]*100

/tmp/ipykernel_2512/3794233381.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Athlete age'] = df['Event dates'].dt.year - df['Athlete year of birth']


0.0

In [ ]:
df['Event dates'].dt.year.unique()

array([2018, 2019, 2016, 2017, 2020, 2021, 2022, 2023, 1997, 1996, 1998,
       1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009,
       2010, 2011, 2012, 2013, 2014, 2015, 1861, 1870, 1874, 1875, 1876,
       1878, 1879, 1880, 1881, 1882, 1884, 1887, 1888, 1927, 1928, 1929,
       1932, 1959, 1963, 1964, 1965, 1966, 1967, 1968, 1969, 1970, 1971,
       1972, 1973, 1974, 1975, 1976, 1977, 1978, 1979, 1980, 1981, 1982,
       1984, 1983, 1985, 1986, 1987, 1988, 1989, 1990, 1991, 1992, 1993,
       1994, 1995], dtype=int32)

In [ ]:
promedios = pd.DataFrame[{
    'prom. categoria edad' :
}]

In [ ]:
# APLICAR SIEMPRE Y CUANDO CANTIDAD DE VALORES NULOS SEA < 5% DEL DATAFRAME
if df.loc[(df['Athlete year of birth'].isna()) & (df['Athlete age category'].isna())].shape[0]/df.shape[0]*100 < 5:
  df = df.loc[(df['Athlete year of birth'].isna() == False) & (df['Athlete age category'].isna() == False)]

In [ ]:
df = df.loc[df['Athlete age'] >= 17]

In [ ]:
df['Athlete age'].head()

,Athlete age
55,35.0
56,41.0
57,42.0
58,32.0
59,30.0


In [ ]:
df = df.drop(columns=['Athlete club',
                      'Year of event',
                      'Athlete year of birth',
                      'Athlete age category'])

In [ ]:
df[['Event type','Event stages']] = np.nan

In [ ]:
matches = ['mi', 'miles', 'mil', 'km', 'k']
time_matches = ['h', 'd',':']

def clase_evento(x):
    if not isinstance(x, str):
        return np.nan

    x_lower = x.lower()

    if any(m in x_lower for m in matches):
        return 'Distance'
    if any(t in x_lower for t in time_matches):
        return 'Timed'

    return 'Unknown'

df['Event type'] = df['Event distance/length'].apply(clase_evento)

In [ ]:
df[df['Event type'].isna()].shape[0]

0

In [ ]:
df.head(3)

,Event dates,Event name,Event distance/length,Event number of finishers,Athlete performance,Athlete country,Athlete gender,Athlete average speed,Athlete ID,Athlete age,Event type,Event stages
55,2018-01-06,Yankee Springs 50 Mile Winter Challenge (USA),50mi,9,9:53:05 h,USA,M,8.141,55,35.0,Distance,NaN
56,2018-01-06,Yankee Springs 50 Mile Winter Challenge (USA),50mi,9,11:09:35 h,CAN,F,7.211,56,41.0,Distance,NaN
57,2018-01-06,Yankee Springs 50 Mile Winter Challenge (USA),50mi,9,11:33:00 h,CAN,M,6.967,57,42.0,Distance,NaN


In [ ]:
df['Event distance/length'].loc[df['Event distance/length'].str.find('Etappen') > -1].unique()

array(['150km/3Etappen', '110mi/2Etappen', '81mi/2Etappen',
       '143mi/5Etappen', '75mi/3Etappen', '163mi/6Etappen',
       '170mi/6Etappen', '146mi/5Etappen', '150mi/6Etappen',
       '155mi/6Etappen', '147mi/6Etappen', '148mi/5Etappen',
       '170km/5Etappen', '165mi/6Etappen', '171mi/6Etappen',
       '158mi/4Etappen', '100mi/3Etappen', '100mi/2Etappen',
       '200mi/4Etappen', '120km/3Etappen', '130km/3Etappen',
       '142km/3Etappen', '3204.7mi/70Etappen', '3303mi/80Etappen',
       '230mi/7Etappen', '160mi/6Etappen', '220km/5Etappen',
       '169mi/6Etappen', '124mi/3Etappen', '175km/4Etappen',
       '3423mi/83Etappen', '3554mi/78Etappen', '100km/2Etappen',
       '156mi/3Etappen', '60mi/2Etappen', '150mi/3Etappen',
       '558.4km/8Etappen', '2935.8mi/64Etappen', '2912.4mi/64Etappen',
       '2925.7mi/64Etappen', '2906.1mi/64Etappen'], dtype=object)

In [ ]:
df['Event stages'] = df['Event distance/length'].apply(lambda x : re.search(r'\d+\.?\d*', x[-9:]).group() if x.find('Etappen') > -1 else 1)
if df.loc[pd.to_numeric(df['Event stages'], errors='coerce').isna()].shape[0] == 0:
  df['Event stages'] = pd.to_numeric(df['Event stages'], errors='coerce')

In [174]:
df['Event distance/length'] = df['Event distance/length'].apply(lambda x : re.sub(r'/.*', '', x) if re.search(r'Etappen', x, re.IGNORECASE) else x)
df['Event distance/length'].unique()

array(['50mi', '50km', '28mi', '33mi', '100mi', '100km', '60mi', '6h',
       '9h', '24h', '12h', '40mi', '48h', '10h', '51km', '51.4mi', '32h',
       '150km', '11h', '7h', '60km', '35h', '40mile', '42.4mi', '29.3mi',
       '116mi', '34mi', '55km', '75mi', '1000mi', '350mi', '130mi',
       '47km', '52km', '71mi', '30h', '15h', '72h', '28.2mi', '52mi',
       '39mi', '110mi', '8h', '118mi', '135mi', '54km', '108km', '200mi',
       '150mi', '46.5mi', '30mi', '52.4mi', '130km', '101km', '6d', '17h',
       '10d', '32.5mi', '57km', '88km', '33.5mi', '70km', '44.4mi',
       '35mi', '51mi', '66km', '68mi', '155mi', '5h', '32mi', '22h',
       '70.5mi', '81mi', '38mi', '112mi', '36h', '104.8mi', '96h',
       '250mi', '151mi', '888km', '56mi', '31mi', '13h', '56km', '106mi',
       '4d', '43mi', '55mi', '53km', '53mi', '33.4mi', '102mi', '85km',
       '79mi', '54mi', '500km', '5d', '103mi', '51.5mi', '64km', '15:30h',
       '3100mi', '1000km', '143mi', '45km', '105km', '110km', '70mi',

In [ ]:
df['Event distance/length'].loc[df['Event distance/length'].str.find('km') <= -1].unique()

array(['50mi', '28mi', '33mi', '100mi', '60mi', '6h', '9h', '24h', '12h',
       '40mi', '48h', '10h', '51.4mi', '32h', '11h', '7h', '35h',
       '40mile', '42.4mi', '29.3mi', '116mi', '34mi', '75mi', '1000mi',
       '350mi', '130mi', '71mi', '30h', '15h', '72h', '28.2mi', '52mi',
       '39mi', '110mi/2Etappen', '8h', '118mi', '135mi', '200mi', '150mi',
       '46.5mi', '30mi', '52.4mi', '6d', '17h', '10d', '32.5mi', '33.5mi',
       '44.4mi', '35mi', '51mi', '68mi', '155mi', '5h', '32mi', '22h',
       '70.5mi', '81mi/2Etappen', '38mi', '112mi', '36h', '104.8mi',
       '96h', '250mi', '151mi', '56mi', '31mi', '13h', '106mi', '4d',
       '43mi', '55mi', '53mi', '33.4mi', '102mi', '79mi', '54mi', '5d',
       '103mi', '51.5mi', '15:30h', '3100mi', '143mi/5Etappen', '70mi',
       '9:11h', '45mi', '202mi', '46mi', '71.1mi', '86h', '31.5mi',
       '206mi', '75mi/3Etappen', '99mi', '36mi', '90mi', '163mi/6Etappen',
       '68h', '112.8mi', '72mi', '238mi', '170mi/6Etappen', '55h', '8

####ESTO ESTA MALO!!!!!!!!!!!!!!!!

In [ ]:
#FOR REUTILIZA VARIABLE X
'''
def millas_a_km(x):
    matches = ['mi','miles','mil']
    if isinstance(x, str) and any(x.lower() in matches for x in matches):
        miles = float(re.search(r'\d+\.?\d*', x).group())
        if miles > 0:
            km = round(miles * 1.60934)
            return f"{km}km"
    return x

df['Event distance/length'] = df['Event distance/length'].apply(millas_a_km)
'''

In [ ]:
df[['Athlete performance','Athlete average speed']].loc[df['Athlete performance'].str.find('h') <= -1]

,Athlete performance,Athlete average speed
1109,56.648 km,9.441
1110,51.499 km,8.583
1111,46.349 km,7.725
1112,46.349 km,7.725
1113,46.349 km,7.725
...,...,...
7459134,56.327 km,4694.0
7459136,53.237 km,4436.0
7459137,53.108 km,4426.0
7459138,51.499 km,4292.0


In [ ]:
lista_p = df['Athlete ID'].loc[df['Athlete performance'].str.find('h') <= -1]
lista_p.head(3)

,Athlete ID
1109,1107
1110,1108
1111,1109


In [ ]:
filas_con_string = df[df['Athlete performance'].apply(type) == str]
filas_con_string.shape[0]

327307

In [ ]:
lista_avg = df[['Athlete ID','Athlete average speed']].loc[pd.to_numeric(df['Athlete average speed'], errors='coerce').isna()]
lista_avg

,Athlete ID,Athlete average speed
20137,19334,21:00:00
20138,19335,20:00:00
20139,19336,19:00:00
20140,19337,17:00:00
20141,19338,12:00:00
...,...,...
6736060,19038,09:00:00
6736061,379764,08:00:00
6736062,3884,08:00:00
6736063,1262098,07:00:00


In [ ]:
pd.to_numeric(df['Athlete average speed'], errors='ignore')

/tmp/ipykernel_616/4195846785.py:1: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  pd.to_numeric(df['Athlete average speed'], errors='ignore')


,Athlete average speed
55,8.141
56,7.211
57,6.967
58,6.914
59,6.738
...,...
7461181,6709.0
7461182,6690.0
7461183,6674.0
7461184,6674.0


In [ ]:
#CORREGIR CALCULO
'''
def km_a_hr(x, y):
  if isinstance(x, str) and not 'h' in x:
    num = float(re.search(r'\d+\.?\d*', x).group())
    n = float(y)
    if n > 0:
      time = num/n
      hr = int(time)
      m = int((time - hr) * 60)
      s = int(round(((time - hr) * 60 - m) * 60))
      return f'{hr}:{m}:{s} h'
  return x

df['Athlete performance'] = df.apply(lambda row: km_a_hr(row['Athlete performance'], row['Athlete average speed']), axis=1)
'''

ValueError: could not convert string to float: '21:00:00'

In [ ]:
#Verificar cambios
df['Athlete performance'].loc[df['Athlete ID'].isin(lista)]

,Athlete performance
151107,4:46:26 h
151110,4:49:36 h
151111,4:50:02 h
151118,5:07:22 h
151119,5:09:16 h
...,...
6923498,7:48:38 h
6943599,7:10:00 h
6943600,7:26:40 h
6966461,7:49:27 h


In [ ]:
df.columns.tolist()

['Event dates',
 'Event name',
 'Event distance/length',
 'Event number of finishers',
 'Athlete performance',
 'Athlete country',
 'Athlete gender',
 'Athlete average speed',
 'Athlete ID',
 'Athlete age']

In [ ]:
df = df[[
 'Athlete ID',
 'Athlete country',
 'Athlete age',
 'Athlete gender',
 'Athlete performance',
 'Athlete average speed',
 'Event dates',
 'Event name',
 'Event distance/length',
 'Event number of finishers']]

In [ ]:
df.duplicated().sum()

np.int64(0)

In [ ]:
nulos = pd.DataFrame({
    "nulos": df.isna().sum(),
    "pct": df.isna().mean() * 100,
}).sort_values("pct", ascending=False)

nulos

,nulos,pct
Athlete average speed,180,0.012974
Athlete ID,0,0.000000
Athlete age,0,0.000000
Athlete country,0,0.000000
Athlete gender,0,0.000000
Athlete performance,0,0.000000
Event dates,0,0.000000
Event name,0,0.000000
Event distance/length,0,0.000000
Event number of finishers,0,0.000000


In [ ]:
df.head()

,Athlete ID,Athlete country,Athlete age,Athlete gender,Athlete performance,Athlete average speed,Event dates,Event name,Event distance/length,Event number of finishers
151102,1489,HUN,38.0,M,4:09:59 h,14.713,2018-04-08,Supermaraton Zagreb-Cazma (CRO),61.3km,83
151103,23310,CRO,33.0,M,4:12:26 h,14.57,2018-04-08,Supermaraton Zagreb-Cazma (CRO),61.3km,83
151104,100946,SLO,42.0,M,4:30:25 h,13.601,2018-04-08,Supermaraton Zagreb-Cazma (CRO),61.3km,83
151105,120173,CRO,27.0,M,4:34:55 h,13.379,2018-04-08,Supermaraton Zagreb-Cazma (CRO),61.3km,83
151106,120174,CRO,21.0,M,4:43:08 h,12.99,2018-04-08,Supermaraton Zagreb-Cazma (CRO),61.3km,83


In [ ]:
df.shape[0]-df['Athlete country'].loc[df['Athlete country'].isin(countries['Country'])].shape[0]

0

In [ ]:
df['Athlete country'].unique()

array(['HUN', 'CRO', 'SLO', 'BIH', 'SRB', 'FRA', 'AUT', 'GER', 'ITA',
       'RUS', 'SUI', 'POL', 'ROU', 'UKR', 'LTU', 'MKD', 'BLR', 'ESP',
       'CZE', 'SGP', 'FIN', 'BUL', 'LAT', 'SWE', 'IRL', 'SVK', 'GBR',
       'USA', 'NOR', 'MNE', 'MRI', 'NZL', 'POR', 'BEL', 'ISR', 'EST',
       'KOR', 'JPN', 'DEN', 'GRE', 'MAS', 'CYP', 'NED', 'COL', 'INA',
       'BRA', 'RSA', 'AUS', 'TUR', 'CAN', 'ARG', 'IND', 'URU', 'MEX',
       'CPV', 'TPE', 'LBN', 'TAN', 'ISL', 'BOT', 'EGY', 'THA', 'HKG',
       'MDA', 'ALG', 'CHN', 'OMA', 'QAT', 'PER', 'MAC', 'NEP', 'MDV',
       'MAR'], dtype=object)

In [ ]:
first_year = df['Event dates'].dt.year.min()
last_year = df['Event dates'].dt.year.max()

In [ ]:
first_year, last_year

(1979, 2022)

In [ ]:
df.to_csv(f'ULTRAMARATONES_{pais}_({first_year}-{last_year}).csv', index=False)